# 🎯 Lab W5-1 — Threshold Policy ภายใต้ต้นทุนที่ไม่สมมาตร

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 5 — Data Mining I**

Lab นี้ใช้คู่กับสื่อจำลอง **Threshold Policy Studio** (`/sims/threshold-policy`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. อธิบายได้ว่าเหตุใด **accuracy จึงโกหก** เมื่อข้อมูลไม่สมดุล
2. คำนวณ **ต้นทุนรวมของความผิดพลาด** และใช้เป็นเกณฑ์เลือก threshold
3. แสดงว่า **F1 ก็ยังตอบผิด** เมื่อ FP กับ FN มีราคาต่างกัน
4. ออกแบบ **นโยบายสามระดับ** ที่ทำงานได้จริงภายใต้เพดานกำลังคน

## ข้อมูล
`fraud_scored.csv` — ธุรกรรม 20,000 รายการ พร้อมคะแนนความเสี่ยงจากโมเดลที่ฝึกเสร็จแล้ว

**โจทย์ไม่ใช่การสร้างโมเดล** โมเดลเสร็จแล้ว งานที่เหลือคือสิ่งที่ยากกว่า —
ต้องขีดเส้นที่ค่าเท่าไรจึงจะเรียกว่า "น่าสงสัย"

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week05/fraud_scored.csv")
df = pd.read_csv(URL)

COST_FN = 8000    # ปล่อยรายการทุจริตผ่าน — ธนาคารต้องคืนเงินลูกค้า
COST_FP = 300     # แจ้งเตือนผิด — ค่าแรงตรวจสอบ + ความรำคาญของลูกค้า
CAPACITY = 7200   # ทีมตรวจสอบรับได้ 120 เคส/วัน × 60 วัน

n = len(df)
pos = int(df.is_fraud.sum())
print(f"ธุรกรรมทั้งหมด : {n:,}")
print(f"ทุจริตจริง     : {pos:,} ({pos/n*100:.2f}%)")
print(f"ต้นทุน FN : {COST_FN:,} บาท  ·  FP : {COST_FP:,} บาท  "
      f"(ไม่สมมาตร {COST_FN/COST_FP:.1f} เท่า)")
df.head(5)

## ส่วนที่ 1 — กับดัก Accuracy

ก่อนแตะโมเดลเลย ให้ดูว่า "ไม่ทำอะไรเลย" ได้คะแนนเท่าไร

### 🧑‍💻 งานที่ 1
คำนวณ accuracy ของตัวจำแนกที่ **ทายว่า "ไม่ทุจริต" ทุกรายการ**
แล้วคำนวณต้นทุนรวมที่ธนาคารต้องจ่ายจากนโยบายนี้

*เฉลยที่ถูกต้อง: accuracy 98.33% · ต้นทุน 2,664,000 บาท*

In [ ]:
base_acc = (n - pos) / n
do_nothing_cost = pos * COST_FN

print(f"ตัวจำแนกที่ไม่ต้องมีโมเดลเลย")
print(f"  accuracy    : {base_acc*100:.2f}%")
print(f"  ต้นทุนรวม   : {do_nothing_cost:,} บาท")
print(f"  จับได้       : 0 รายการ จาก {pos:,} รายการ")
print("""
→ ถ้ารายงานผลด้วย accuracy อย่างเดียว โมเดลที่ 'ไม่ทำอะไรเลย'
  ก็ผ่านเกณฑ์ 95% ที่หลายองค์กรตั้งไว้ได้สบาย ๆ
  ตัววัดนี้จึงไร้ความหมายโดยสิ้นเชิงเมื่อข้อมูลไม่สมดุล
""")

## ส่วนที่ 2 — กวาดทุก threshold

### 🧑‍💻 งานที่ 2
เขียนฟังก์ชัน `evaluate(t)` ที่คืน dict ประกอบด้วย
`tp, fp, fn, tn, alerts, precision, recall, accuracy, f1, cost`

แล้วสร้าง `DataFrame` ชื่อ `sweep` ที่มีผลของทุก threshold ตั้งแต่ 0.01 ถึง 0.99 (ก้าวละ 0.01)

*เฉลยที่ถูกต้อง: ที่ t = 0.50 → tp = 286, fp = 119, fn = 47, ต้นทุน 411,700 บาท*

In [ ]:
score = df.risk_score.values
truth = df.is_fraud.values


def evaluate(t: float) -> dict:
    pred = score >= t
    tp = int((pred & (truth == 1)).sum())
    fp = int((pred & (truth == 0)).sum())
    fn = pos - tp
    tn = n - pos - fp
    alerts = tp + fp
    return {
        "t": round(t, 2), "tp": tp, "fp": fp, "fn": fn, "tn": tn, "alerts": alerts,
        "precision": tp / alerts if alerts else np.nan,
        "recall": tp / pos,
        "accuracy": (tp + tn) / n,
        "f1": 2 * tp / (2 * tp + fp + fn) if tp else 0.0,
        "cost": fn * COST_FN + fp * COST_FP,
    }


sweep = pd.DataFrame([evaluate(k / 100) for k in range(1, 100)]).set_index("t")

print(sweep.loc[[0.10, 0.20, 0.30, 0.40, 0.42, 0.50, 0.55, 0.57, 0.70, 0.90]].to_string())

## ส่วนที่ 3 — เกณฑ์คนละแบบ ชี้ไปคนละจุด

### 🧑‍💻 งานที่ 3
หา threshold ที่ดีที่สุดตามเกณฑ์ 3 แบบ แล้วสร้างตารางเปรียบเทียบ

1. accuracy สูงสุด
2. F1 สูงสุด
3. ต้นทุนรวมต่ำสุด

แล้วตอบว่าการเลือกตามเกณฑ์ 1 และ 2 ทำให้ธนาคารเสียเงินเพิ่มเท่าไรเทียบกับเกณฑ์ 3

*เฉลยที่ถูกต้อง: acc สูงสุดที่ 0.57 · F1 สูงสุดที่ 0.55 · ต้นทุนต่ำสุดที่ 0.42*

In [ ]:
picks = {
    "accuracy สูงสุด": sweep.accuracy.idxmax(),
    "F1 สูงสุด": sweep.f1.idxmax(),
    "ต้นทุนรวมต่ำสุด ✅": sweep.cost.idxmin(),
}

cmp = sweep.loc[list(picks.values()), ["accuracy", "precision", "recall", "f1", "alerts", "cost"]]
cmp.index = [f"{k}  (t={v:.2f})" for k, v in picks.items()]
print(cmp.to_string())

best_t = sweep.cost.idxmin()
best_cost = sweep.cost.min()
print(f"\nต้นทุนต่ำสุด = {best_cost:,} บาท ที่ threshold {best_t:.2f}")
for name, t in picks.items():
    extra = sweep.loc[t, "cost"] - best_cost
    print(f"  เลือกตาม {name:<20} → เสียเพิ่ม {extra:>10,.0f} บาท")

print(f"\nเทียบกับการไม่ทำอะไรเลย ประหยัดได้ {do_nothing_cost - best_cost:,} บาท")

> **เหตุใด F1 จึงยังตอบผิด**
>
> F1 คือค่าเฉลี่ยฮาร์มอนิกของ precision กับ recall ซึ่งแปลว่ามัน **ถ่วงน้ำหนักทั้งสองเท่ากัน**
> การถ่วงเท่ากันคือการสมมติว่า FP กับ FN แพงเท่ากัน
>
> ในโจทย์นี้ FN แพงกว่า FP ถึง 26.7 เท่า F1 จึงเลือกจุดที่ระมัดระวังเกินไป
> และปล่อยรายการทุจริตผ่านมากกว่าที่ควร

## ส่วนที่ 4 — เพดานกำลังคน

### 🧑‍💻 งานที่ 4
หา threshold ที่ **ต่ำที่สุด** ที่จำนวนเคสยังไม่เกินเพดาน 7,200 เคส
แล้วตอบว่า

1. ถ้าอยากได้ recall = 100% ต้องใช้ threshold เท่าไร และต้องตรวจกี่เคส
2. เพดานกำลังคนเป็นข้อจำกัดที่ผูกมัดจริงหรือไม่ในกรณีนี้ เพราะเหตุใด

*เฉลยที่ถูกต้อง: recall 100% ต้องใช้ t ≤ 0.16 → 7,959 เคส ซึ่งเกินเพดาน 759 เคส*

In [ ]:
feasible = sweep[sweep.alerts <= CAPACITY]
print(f"threshold ต่ำสุดที่ยังไม่เกินเพดาน : {feasible.index.min():.2f} "
      f"({feasible.alerts.max():,} เคส)")

full_recall = sweep[sweep.recall >= 1.0]
t_full = full_recall.index.max()
print(f"\nrecall = 100% ต้องใช้ t ≤ {t_full:.2f} → ต้องตรวจ {sweep.loc[t_full,'alerts']:,} เคส "
      f"(เกินเพดาน {sweep.loc[t_full,'alerts']-CAPACITY:,} เคส)")
print(f"ต้นทุนของนโยบาย recall 100% = {sweep.loc[t_full,'cost']:,} บาท")

print(f"\nจุดที่ต้นทุนต่ำสุด t={best_t:.2f} ใช้เพียง {sweep.loc[best_t,'alerts']:,} เคส "
      f"(เหลือกำลัง {CAPACITY - sweep.loc[best_t,'alerts']:,} เคส)")

print("""
คำตอบข้อ 2
----------
เพดานกำลังคน 'ไม่ใช่' ข้อจำกัดที่ผูกมัดในกรณีนี้ — เศรษฐศาสตร์บีบก่อน
จุดที่ต้นทุนต่ำสุดใช้เคสเพียง 700 เคส ห่างจากเพดาน 7,200 เคสอยู่มาก

นักศึกษาส่วนใหญ่คาดว่าเพดานจะเป็นตัวบีบ เพราะคุ้นกับการคิดว่า
'ยิ่งจับได้มากยิ่งดี แต่คนไม่พอ' ความจริงคือการไล่ recall ให้ถึง 100%
ทำให้ต้องตรวจ FP จำนวนมหาศาลจนต้นทุนรวมแพงกว่าการยอมพลาดบางรายการ

ข้อจำกัดที่ 'ดูเหมือน' ผูกมัด กับข้อจำกัดที่ผูกมัดจริง มักไม่ใช่ตัวเดียวกัน
ต้องพิสูจน์ด้วยตัวเลขเสมอ ไม่ใช่เดาจากสัญชาตญาณ
""")

## ส่วนที่ 5 — ความไวต่ออัตราส่วนต้นทุน

### 🧑‍💻 งานที่ 5
ต้นทุน 8,000 กับ 300 เป็นค่าประมาณ ผู้บริหารอาจเถียงว่าไม่ตรง
ให้แสดงว่า threshold ที่ดีที่สุดขยับไปอย่างไรเมื่ออัตราส่วน `COST_FN / COST_FP`
เปลี่ยนจาก 1 เท่า ไปจนถึง 100 เท่า

แล้วตอบว่าข้อสรุปของ Lab นี้ยังยืนอยู่หรือไม่ถ้าตัวเลขต้นทุนคลาดเคลื่อน

In [ ]:
rows = []
for ratio in [1, 2, 5, 10, 20, 26.7, 50, 100]:
    cost_fn = COST_FP * ratio
    c = sweep.fn * cost_fn + sweep.fp * COST_FP
    rows.append({"อัตราส่วน FN:FP": ratio, "threshold ที่ดีที่สุด": c.idxmin(),
                 "recall ที่จุดนั้น": sweep.loc[c.idxmin(), "recall"],
                 "เคสที่ต้องตรวจ": int(sweep.loc[c.idxmin(), "alerts"])})

sens = pd.DataFrame(rows).set_index("อัตราส่วน FN:FP")
print(sens.to_string())

print("""
ข้อสรุป
-------
threshold ที่ดีที่สุดลดลงอย่างเป็นระบบเมื่อ FN แพงขึ้นเมื่อเทียบกับ FP
ซึ่งสมเหตุสมผล: ยิ่งการพลาดแพง ยิ่งต้องหว่านแหกว้าง

ข้อสรุปหลักของ Lab ยังยืน — แม้อัตราส่วนจะคลาดเคลื่อนไปสองสามเท่า
threshold ที่ดีที่สุดก็ยังต่ำกว่าจุดที่ accuracy หรือ F1 สูงสุดอยู่ดี
สิ่งที่เปลี่ยนคือ 'ตัวเลขที่แน่นอน' ไม่ใช่ 'ทิศทางของคำตอบ'

นี่คือเหตุผลที่ต้องรายงานผลเป็นช่วงเสมอ ไม่ใช่ตัวเลขเดียว
และต้องแนบการวิเคราะห์ความไว (sensitivity analysis) ไปกับข้อเสนอทุกครั้ง
""")

## ส่วนที่ 6 — นโยบายสามระดับ

### 🧑‍💻 งานที่ 6
ในโลกจริงแทบไม่มีใครใช้ threshold เดียว ให้ออกแบบนโยบายสามระดับด้วยสอง threshold

| ช่วงคะแนน | การกระทำ | ต้นทุนต่อเคส | ผลต่อการทุจริต |
|---|---|---|---|
| ต่ำกว่า `t_low` | ปล่อยผ่านอัตโนมัติ | 0 | พลาดทุกรายการ (FN เต็มราคา) |
| ระหว่าง `t_low` กับ `t_high` | ส่งให้คนตรวจ | 300 บาท | คนตรวจพลาด 15% ของรายการทุจริต |
| สูงกว่า `t_high` | ระงับทันทีแล้วโทรยืนยัน | 800 บาท | หยุดได้ทุกรายการ |

หาค่า `t_low` และ `t_high` ที่ทำให้ต้นทุนรวมต่ำที่สุด
โดยจำนวนเคสที่ส่งให้คนตรวจต้องไม่เกินเพดาน 7,200 เคส

> **จุดที่ต้องคิดก่อนเขียนโค้ด** การระงับทันทีแพงกว่าการให้คนตรวจเกือบสามเท่า
> แล้วเหตุใดจึงยังคุ้มที่จะระงับในบางช่วงคะแนน

In [ ]:
COST_BLOCK = 800    # ค่าเสียโอกาส + ความไม่พอใจของลูกค้าที่ถูกระงับ
REVIEW_MISS = 0.15  # คนตรวจพลาดรายการทุจริต 15%

best = None
for lo in np.arange(0.05, 0.95, 0.01):
    for hi in np.arange(lo + 0.01, 1.00, 0.01):
        auto_pass = score < lo
        review = (score >= lo) & (score < hi)
        block = score >= hi

        n_review = int(review.sum())
        if n_review > CAPACITY:
            continue

        missed_auto = int((auto_pass & (truth == 1)).sum())          # พลาดทั้งหมด
        missed_review = (review & (truth == 1)).sum() * REVIEW_MISS  # คนตรวจพลาด 15%

        cost = ((missed_auto + missed_review) * COST_FN
                + n_review * COST_FP
                + int(block.sum()) * COST_BLOCK)
        if best is None or cost < best["cost"]:
            best = {"t_low": round(lo, 2), "t_high": round(hi, 2), "cost": round(cost),
                    "n_review": n_review, "n_block": int(block.sum()),
                    "missed_auto": missed_auto, "missed_review": round(missed_review, 1),
                    "caught": int(((review | block) & (truth == 1)).sum())}

print("นโยบายสามระดับที่ดีที่สุด")
for k, v in best.items():
    print(f"  {k:<14} : {v:,}")
print(f"\n  ส่งเข้าระบบตรวจ {best['caught']:,} จาก {pos:,} รายการ "
      f"({best['caught']/pos*100:.2f}%)")

# เทียบกับนโยบาย threshold เดียวภายใต้สมมติฐานเดียวกัน (คนตรวจพลาด 15%)
single = sweep.fn * COST_FN + sweep.tp * REVIEW_MISS * COST_FN + sweep.alerts * COST_FP
print(f"  นโยบาย threshold เดียวที่ดีที่สุดภายใต้สมมติฐานเดียวกัน: "
      f"{single.min():,.0f} บาท ที่ t={single.idxmin():.2f}")
print(f"  นโยบายสามระดับประหยัดกว่า {single.min()-best['cost']:,.0f} บาท")

print("""
เหตุใดการระงับทันทีจึงคุ้มในบางช่วง
------------------------------------
ต้นทุนคาดหวังต่อหนึ่งเคส เมื่อ p = ความน่าจะเป็นที่เคสนั้นทุจริตจริง
  ส่งให้คนตรวจ : 300 + p × 0.15 × 8,000 = 300 + 1,200p
  ระงับทันที   : 800 (คงที่ ไม่ขึ้นกับ p)

สองเส้นตัดกันที่ 300 + 1,200p = 800  →  p = 0.4167

แปลว่าเมื่อใดที่ precision ของช่วงคะแนนนั้นเกิน 41.67% การระงับทันทีถูกกว่า
แม้ราคาป้ายจะแพงกว่าเกือบสามเท่า เพราะมันตัดความเสี่ยงที่คนตรวจจะพลาดออกไปทั้งหมด

นี่คือเหตุผลที่นโยบายจริงมีหลายระดับเสมอ — ต้นทุนที่เหมาะสมของแต่ละการกระทำ
ขึ้นกับความน่าจะเป็น ไม่ใช่ค่าคงที่ที่ใช้ได้กับทุกเคส""")

## ส่วนที่ 7 — เขียนข้อเสนอ

### 🧑‍💻 งานที่ 7 (เขียนเป็นข้อความ)

เขียนข้อเสนอความยาวไม่เกินหนึ่งหน้าถึงผู้บริหาร ประกอบด้วย

1. **threshold ที่เสนอ** พร้อมเหตุผลที่อ้างอิงต้นทุน ไม่ใช่อ้างอิง accuracy
2. **สิ่งที่ต้องยอมแลก** — ระบุให้ชัดว่าจะยังพลาดกี่รายการต่อ 60 วัน และคิดเป็นเงินเท่าไร
3. **ตัวเลขที่ต้องเฝ้าดูหลังใช้งานจริง** อย่างน้อย 3 ตัว พร้อมเกณฑ์ที่ต้องกลับมาทบทวน
4. **สิ่งที่ Lab นี้ยังไม่ได้ตอบ** อย่างน้อย 2 ข้อ

ข้อ 4 สำคัญที่สุด — ลองคิดถึงเรื่องที่ข้อมูลชุดนี้บอกไม่ได้เลย

In [ ]:
print(f"""ตัวอย่างคำตอบข้อ 4 — สิ่งที่ Lab นี้ยังไม่ได้ตอบ
------------------------------------------------
1. ต้นทุน FN 8,000 บาท เป็นค่าเฉลี่ย แต่ความจริงขึ้นกับมูลค่าธุรกรรม
   การพลาดรายการ 500 บาท กับพลาดรายการ 500,000 บาท ไม่ควรมีน้ำหนักเท่ากัน
   นโยบายที่ดีกว่าคือให้ threshold ขึ้นกับ amount ด้วย ไม่ใช่ค่าคงที่ค่าเดียว

2. ไม่มีต้นทุนของ 'ความเสียหายต่อความไว้วางใจ' อยู่ในสมการเลย
   ลูกค้าที่ถูกระงับบัตรผิด ๆ ระหว่างเดินทางอาจเลิกใช้บริการถาวร
   ซึ่งแพงกว่า 300 บาทมาก แต่วัดยากจึงมักถูกละไว้ — การละไว้ไม่ได้แปลว่าเป็นศูนย์

3. โจรปรับตัวได้ เมื่อรู้ว่าเส้นแบ่งอยู่ตรงไหน พฤติกรรมจะเลื่อนไปหลบใต้เส้น
   คะแนนความเสี่ยงในอนาคตจึงไม่กระจายตัวเหมือนในข้อมูลชุดนี้
   ต้องมีรอบทบทวน threshold และต้องไม่เปิดเผยค่าที่ใช้จริงออกนอกองค์กร

4. ข้อมูลชุดนี้มีเฉพาะรายการที่ 'ผ่านเข้ามาในระบบ' แล้ว
   เราไม่มีทางรู้เลยว่ามีการทุจริตกี่รายการที่ระบบก่อนหน้าปัดตกไปแล้ว
   ตัวเลข 1.67% จึงเป็นอัตราที่สังเกตได้ ไม่ใช่อัตราที่แท้จริง
""")

---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — แสดงกับดัก accuracy ด้วยตัวเลข | 2 |
| งานที่ 2 — ฟังก์ชันประเมินและตารางกวาด threshold | 3 |
| งานที่ 3 — เปรียบเทียบ 3 เกณฑ์และคำนวณส่วนต่างต้นทุน | 3 |
| งานที่ 4 — วิเคราะห์เพดานกำลังคนและตอบว่าผูกมัดจริงหรือไม่ | 3 |
| งานที่ 5 — วิเคราะห์ความไวต่ออัตราส่วนต้นทุน | 3 |
| งานที่ 6 — นโยบายสามระดับที่ทำได้จริง | 3 |
| งานที่ 7 — ข้อเสนอถึงผู้บริหาร โดยเฉพาะข้อจำกัดที่ระบุได้ | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/threshold-policy`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง